# APD Layer 3 — Kaggle scheduled shift

Generates pending cells from the **non-Pollinations** slice of the
production grid using `diffusers` on Kaggle's free T4 GPU:

* SD 1.5 / SDXL / SD 3.5 Medium (main grid)
* SD 2.1 / Playground 2.5 / Kandinsky 3 / AltDiffusion-m18 (robustness grid)
* Indigenous-language cells from the robustness grid for the 4 main models
* Optional FLUX cross-validation 5% sample (DESIGN.md §4.2)
* Optional FLUX *full* coverage if Layer 1+2 cannot keep up (D-026 fallback)

Each cell is generated + classified inline; the shard parquet carries the
full 24-column schema (D-028) so downstream merge is one-shot.

**To schedule on Kaggle**: Save & Run All → ⋯ → Schedule a run → daily.
Each scheduled run consumes ~12 GPU-hours from the 30 h/week free quota.
Set `GH_TOKEN_SECRET` to a Kaggle secret holding a fine-grained PAT with
**Contents: Write** on `hlaverde/apd-audit` so the final cell can push.

In [ ]:
# === Configuration — edit these lines for the shift ===
REPO_URL = "https://github.com/hlaverde/apd-audit.git"
BUDGET   = 400        # cells per shift; T4 SDXL ~12s/img → 400 imgs ≈ 1.5h
TIME_BUDGET_S = 11 * 3600  # 11h; Kaggle hard caps at 12h per run

# Which slices to consume in this shift. Toggle to True/False.
INCLUDE_SD_MAIN      = True   # SD 1.5, SDXL, SD 3.5 Medium (main grid)
INCLUDE_ROBUSTNESS   = True   # SD 2.1, Playground 2.5, Kandinsky 3, AltDiffusion-m18
INCLUDE_INDIGENOUS   = True   # robustness × {qu, gn} indigenous languages
INCLUDE_FLUX_FALLBACK = False # only set True if Pollinations rate-limits N=1 too

# Push configuration. The Kaggle secret name must be exactly this string;
# the value is a GitHub fine-grained PAT with Contents:Write.
GH_TOKEN_SECRET = "APD_PUSH_TOKEN"
GH_USER         = "hlaverde"
DO_PUSH         = True


In [ ]:
# Detect environment, configure persistent storage.
import os, sys, subprocess, pathlib

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB  = "google.colab" in sys.modules
print(f"Kaggle: {IS_KAGGLE}  |  Colab: {IS_COLAB}")

if IS_KAGGLE:
    BASE = pathlib.Path("/kaggle/working/apd-audit")
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive/apd-audit")
else:
    BASE = pathlib.Path.cwd() / "apd-audit"
BASE.parent.mkdir(parents=True, exist_ok=True)
print("Project root:", BASE)

In [ ]:
# Read the GitHub push token from the Kaggle secret store (if available).
GH_TOKEN = ""
if IS_KAGGLE and DO_PUSH:
    from kaggle_secrets import UserSecretsClient
    try:
        GH_TOKEN = UserSecretsClient().get_secret(GH_TOKEN_SECRET)
        print(f"Got push token from secret {GH_TOKEN_SECRET!r} (len={len(GH_TOKEN)}).")
    except Exception as exc:
        print(f"Could not read secret {GH_TOKEN_SECRET!r}: {exc}")
        DO_PUSH = False
elif not IS_KAGGLE and DO_PUSH:
    GH_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    if not GH_TOKEN:
        print("GITHUB_TOKEN not in env; will skip push.")
        DO_PUSH = False

In [ ]:
# Clone or pull the repository. Embed the PAT into the remote so subsequent
# push commands are authenticated.
remote = REPO_URL
if DO_PUSH and GH_TOKEN:
    remote = REPO_URL.replace("https://", f"https://{GH_USER}:{GH_TOKEN}@")

if BASE.exists():
    subprocess.run(["git", "-C", str(BASE), "pull", "--ff-only"], check=True)
    if DO_PUSH and GH_TOKEN:
        subprocess.run(["git", "-C", str(BASE), "remote", "set-url", "origin", remote], check=True)
else:
    subprocess.run(["git", "clone", remote, str(BASE)], check=True)

sys.path.insert(0, str(BASE / "src"))
os.environ["PYTHONPATH"] = str(BASE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repo cloned/updated at", BASE)

In [ ]:
# Install runtime deps (sync, no uv overhead). Always install ml extras —
# Layer 3 is GPU-focused and select_backend will raise if they are missing.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pandas>=2.2", "pyarrow>=15", "numpy>=1.26,<3", "scipy>=1.13",
        "pydantic>=2.7", "pydantic-settings>=2.3", "python-dotenv>=1",
        "requests>=2.32", "httpx>=0.27", "huggingface-hub>=0.25",
        "pillow>=10", "opencv-python>=4.9",
        "skin-tone-classifier>=1.2.3",
        # ml extras — required for SD-family on GPU.
        "torch>=2.2", "diffusers>=0.30", "transformers>=4.42",
        "accelerate>=0.33", "safetensors>=0.4",
    ],
    check=True,
)
print("Deps + ml extras installed.")

In [ ]:
# Build the pending-cell list filtered to this shift's slices.
import itertools
import pandas as pd
from apd.prompts.grid import (
    h5_cells, image_id_of, main_cells, pending_cells, robustness_cells,
    MAIN_MODELS, ROBUSTNESS_LANGUAGES_INDIGENOUS,
)

IMAGES_DIR = BASE / "images" / "main"
META_FILE  = IMAGES_DIR / "metadata.parquet"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

shard_metadata_paths = [META_FILE]
if IMAGES_DIR.exists():
    shard_metadata_paths.extend(
        p for p in IMAGES_DIR.glob("metadata_*.parquet") if p.name != "metadata.parquet"
    )

candidates = []
if INCLUDE_SD_MAIN:
    candidates.append(main_cells())
if INCLUDE_ROBUSTNESS:
    candidates.append(robustness_cells())
# H5 marker grid uses pollinations/flux — only include if FLUX fallback.
if INCLUDE_FLUX_FALLBACK:
    candidates.append(h5_cells())

all_pending = list(pending_cells(itertools.chain(*candidates), shard_metadata_paths))

# Filter by slice toggles.
def keep(cell):
    is_flux = cell.model == "pollinations/flux"
    if is_flux:
        return INCLUDE_FLUX_FALLBACK
    is_sd_main = cell.model in MAIN_MODELS and not is_flux
    if is_sd_main and INCLUDE_SD_MAIN:
        return True
    is_indigenous = cell.language in ROBUSTNESS_LANGUAGES_INDIGENOUS
    if is_indigenous and INCLUDE_INDIGENOUS:
        return True
    if INCLUDE_ROBUSTNESS and not is_sd_main and not is_flux:
        return True
    return False

pending = [c for c in all_pending if keep(c)][:BUDGET]
print(f"Pending after filter: {len(all_pending):>6}")
print(f"This shift will do  : {len(pending):>6} (BUDGET={BUDGET})")
if pending:
    sample_models = {c.model for c in pending[:50]}
    print(f"Sample models       : {sorted(sample_models)}")

In [ ]:
# Generate + classify inline, one cell at a time, with a time budget.
import hashlib, time
import numpy as np
from apd.generate.orchestrator import select_backend, image_path
from apd.classify.consensus import consensus_perla
from apd.classify.face_detect import detect_face
from apd.classify.skin_casco import compute_casco_perla, is_available as casco_available
from apd.classify.skin_ita import compute_ita, ita_to_label, ita_to_perla
from apd.classify.skin_mst import compute_mst, mst_to_perla

casco_on = casco_available()
print(f"CASCo available: {casco_on}")

shift_records = []
shift_start   = time.time()
active_backend = {}  # model -> Backend (cache to avoid re-loading SD pipelines)

def _classify(png_path):
    out = {
        "has_face": False, "n_faces": 0,
        "ita_value": np.nan, "ita_label": "no_face", "ita_perla": np.nan,
        "mst_value": np.nan, "mst_perla": np.nan,
        "casco_perla": np.nan, "perla_consensus": np.nan,
        "n_classifiers": 0, "n_concordant": 0, "concordant_2of3": False,
    }
    face = detect_face(png_path)
    out["has_face"] = bool(face.has_face)
    out["n_faces"] = int(face.n_faces)
    if not face.has_face or face.cropped_bgr is None:
        return out
    patch = face.cropped_bgr
    ita = compute_ita(patch)
    mst = compute_mst(patch)
    ita_p = ita_to_perla(ita)
    mst_p = mst_to_perla(mst)
    casco_p = compute_casco_perla(png_path) if casco_on else None
    cons = consensus_perla([ita_p, mst_p, casco_p])
    out.update({
        "ita_value": float(ita),
        "ita_label": ita_to_label(ita),
        "ita_perla": float(ita_p) if ita_p is not None else np.nan,
        "mst_value": float(mst),
        "mst_perla": float(mst_p) if mst_p is not None else np.nan,
        "casco_perla": float(casco_p) if casco_p is not None else np.nan,
        "perla_consensus": float(cons.perla) if cons.perla is not None else np.nan,
        "n_classifiers": int(cons.n_available),
        "n_concordant": int(cons.n_concordant),
        "concordant_2of3": bool(cons.concordant_2of3),
    })
    return out

for i, cell in enumerate(pending, start=1):
    if time.time() - shift_start > TIME_BUDGET_S:
        print(f"Time budget reached at cell {i-1}/{len(pending)}.")
        break
    if cell.model not in active_backend:
        active_backend[cell.model] = select_backend(cell.model)
        print(f"  loaded backend for {cell.model}")
    backend = active_backend[cell.model]
    started = time.time()
    try:
        result = backend.generate(cell.prompt(), cell.seed)
    except Exception as exc:
        print(f"[{i}/{len(pending)}] FAIL {cell.model} seed={cell.seed}: {exc}")
        continue
    occ_dir = IMAGES_DIR / cell.occupation.replace(" ", "_")
    occ_dir.mkdir(parents=True, exist_ok=True)
    png_path = occ_dir / f"seed_{cell.seed}.png"
    png_path.write_bytes(result.image_bytes)
    classification = _classify(png_path)
    shift_records.append({
        "image_id": image_id_of(cell),
        "model": cell.model,
        "occupation": cell.occupation,
        "language": cell.language,
        "country_proxy": cell.country,
        "seed": cell.seed,
        "prompt": cell.prompt(),
        "path": str(png_path),
        "sha256": result.sha256,
        "backend": result.backend,
        "duration_s": result.duration_s,
        "timestamp": int(time.time()),
        **classification,
    })
    if i % 10 == 0:
        elapsed = time.time() - shift_start
        rate = i / elapsed * 60
        print(f"  [{i}/{len(pending)}] {rate:.1f} imgs/min, elapsed {elapsed/60:.1f} min")

elapsed = time.time() - shift_start
print(f"\nGenerated {len(shift_records)} images in {elapsed/60:.1f} min ({elapsed/max(len(shift_records),1):.1f} s/img).")

In [ ]:
# Write the shard parquet — one file per Kaggle run, named by run id + ts.
if shift_records:
    run_id = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "local") + "_" + str(int(shift_start))
    shard_path = IMAGES_DIR / f"metadata_kaggle_{run_id}.parquet"
    df = pd.DataFrame(shift_records)
    df.to_parquet(shard_path, index=False)
    print(f"Wrote shard: {shard_path} ({len(df)} rows, {len(df.columns)} cols)")
else:
    print("Nothing to write — no cells processed in this shift.")

In [ ]:
# Append a row to docs/COST_LOG.md so cumulative stays $0.00.
if shift_records:
    cost_log = BASE / "docs" / "COST_LOG.md"
    line = (
        f"| {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} | shift Layer-3 Kaggle | "
        f"Kaggle T4 free | {len(shift_records)} imgs / {elapsed/60:.1f} min | $0.00 | **$0.00** |\n"
    )
    with cost_log.open("a", encoding="utf-8") as fh:
        fh.write(line)
    print("Appended to COST_LOG.md.")

In [ ]:
# Push: commit shard + cost log, push to origin/main with rebase if needed.
if DO_PUSH and shift_records and GH_TOKEN:
    subprocess.run(["git", "-C", str(BASE), "config", "user.email", "kaggle-layer3@apd-audit.local"], check=True)
    subprocess.run(["git", "-C", str(BASE), "config", "user.name",  "APD Layer-3 Kaggle bot"], check=True)
    subprocess.run(["git", "-C", str(BASE), "add",
                    "images/main/metadata_kaggle_*.parquet", "docs/COST_LOG.md"], check=False)
    cm = subprocess.run(["git", "-C", str(BASE), "commit", "-m",
                         f"shift: Layer-3 Kaggle +{len(shift_records)} imgs (run {run_id})"], check=False)
    if cm.returncode == 0:
        subprocess.run(["git", "-C", str(BASE), "pull", "--rebase", "--autostash"], check=False)
        push = subprocess.run(["git", "-C", str(BASE), "push"], check=False)
        print(f"Push exit code: {push.returncode}")
    else:
        print("Nothing to commit (commit exit non-zero).")

## End of Layer 3 shift

After this run finishes, the shard `metadata_kaggle_<run_id>.parquet` is
committed and pushed to `origin/main`. The Layer 1 GH Actions cron and the
Layer 2 local async worker will pick it up at their next pending-cells
computation, dedupe via `merge_worker_shards.py`, and adjust their work
accordingly.

**Next manual step**: schedule this notebook in Kaggle (⋯ → Schedule a run
→ daily) so it runs without intervention. The 30 GPU-h/week free quota
covers roughly 12 000 SDXL imgs per week at 8 s/img per GPU-hour.